# Basic Cooling tower

The notebook is used to connect to APS and update the values in APS.  



In [1]:
import sys
import logging
import pandas as pd
import numpy as np
import simcentralconnect
import os
from io import StringIO
import numpy as np
from System.Collections.Generic import Dictionary
from System import Array, String, Double, Object, AggregateException, InvalidOperationException
import json
import plotly.express as px
import statsmodels.api as stm 

In [2]:
#### Connect to SimCentral ####
sc = simcentralconnect.connect().Result

# Instantiate the required interfaces
lm = sc.GetService("ILibraryManager")
sm = sc.GetService("ISimulationManager")
vm = sc.GetService("IVariableManager")
pm = sc.GetService("IParameterManager")
mm = sc.GetService("IModelManager")
cm = sc.GetService("IConnectorManager")
dm = sc.GetService("IDiagramManager")
ssm = sc.GetService("ISnapshotManager")
om = sc.GetService("IOptimizationSetManager")

# API logging is ON by default.  Disabled by changing to 'false'
sc.SetOptions(repr({'Timeout': 10000, 'EnableApiLogging': 'true'}))

SimCentral.Client.Auto loaded from: C:\Program Files\SimSci\SimCentral\Scripting\DotNetFwk


In [3]:
logger_file_path = "./debug.log"
                        

logging.basicConfig(format='[%(asctime)s] : [%(levelname)s] : [%(filename)s:%(lineno)s] : %(message)s',  datefmt='%Y-%m-%d %H:%M:%S',level=logging.DEBUG,handlers=[
        logging.FileHandler(logger_file_path,mode = "a" ),
        logging.StreamHandler()])
logger = logging.getLogger(__name__)

## Main Functions


In [4]:

def open_simulation(sm, sim_name,logger):
    """Helper function to attempt to open a simulation and gracefully handle the case when it doesn't exist."""
    opened = False
    try:
        opened = sm.OpenSimulation(sim_name).Result
    except AggregateException as ex:
        if not isinstance(ex.InnerException, InvalidOperationException) \
            or "simulation doesn't exists" not in ex.InnerException.Message:
            logger.error(f"Simulation '{sim_name}' is not in the repository.  Import '{sim_name}.simx' first")
            raise
    return opened



In [6]:
# Simulation inputs

sim_name  = "basic_flash_drum" 

# Measures we wish to read from APS
measure = Array[String](["FEED.F", "FEED.T"])
uom = Array[String](["kmol/h", "C"])  

# Results we wish to return from APS
sim_measures =  Array[String](["VAP.F", "LIQ.F"])
measures_uom = Array[String](["kmol/h", "kmol/h"])  

In [7]:
opened = open_simulation(sm, sim_name,logger)
logger.info(f"Simulation '{sim_name}' opened")

[2024-11-20 13:13:46] : [INFO] : [3207853750.py:2] : Simulation 'basic_flash_drum' opened


In [8]:
if opened:
    
    FEED_F = 110.0
    FEED_T = 80.0
    values = [FEED_F, FEED_T] 
    value = Array[Double](values)
    set_status = vm.SetVariableValue(sim_name,measure,value,uom, 2000).Result  
    
else:
    logger.error(f"Simulation '{sim_name}' failed to open")
    sys.exit(1)

logger.info(f"Simulation '{sim_name}' completed")

[2024-11-20 13:14:16] : [INFO] : [266095938.py:13] : Simulation 'basic_flash_drum' completed


In [8]:
if opened:    
    # get the simulation results
    results = vm.GetVariableValue(sim_name, sim_measures,measures_uom, 2000).Result    
    # Get Simulation Status
    ss = sm.GetSimulationStatus(sim_name).Result
    try:
        if ss[2]:   
            # Get the results
            VAP_F = results[0] 
            LIQ_F = results[1] 
                
            
            logger.info(f"Input Status: {ss[0]} Specification Status: {ss[1]}, Solve Status: {ss[2]}")
            logger.info(f"The Vapor flow rate is {VAP_F:.2f} and the Liquid flow rate is {LIQ_F:.2f}")
    except Exception as e:
        status = "invalid status"
        logger.error(f"{status}: {e}")
            
                    
else:
    logger.error(f"Simulation '{sim_name}' failed to open")
    sys.exit(1)

logger.info(f"Simulation '{sim_name}' completed")

[2024-11-15 11:20:58] : [INFO] : [2111075141.py:13] : Input Status: True Specification Status: True, Solve Status: True
[2024-11-15 11:20:58] : [INFO] : [2111075141.py:14] : The Vapor flow rate is 35.91 and the Liquid flow rate is 84.09
[2024-11-15 11:20:58] : [INFO] : [2111075141.py:24] : Simulation 'basic_flash_drum' completed
